# TMDB Movie Data Pipeline & Ingestion

### Environment & Authenication

In [1]:
!pip install requests

import requests
import getpass
import sqlite3

print("INFO: Required libraries installed and imported successfully.")

Defaulting to user installation because normal site-packages is not writeable
INFO: Required libraries installed and imported successfully.


In [2]:
access_token = getpass.getpass("Enter your TMDB API access token: ")

headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {access_token}"
}

print("INFO: API access token securely captured and authorization headers configured.")

INFO: API access token securely captured and authorization headers configured.


### Automated Ingestion & Pagination

In [3]:
all_raw_movies = []

for page in range(1, 6):
    URL = f"https://api.themoviedb.org/3/movie/popular?page={page}"
    response = requests.get(URL, headers=headers)
    page_results = response.json().get("results", [])
    
    all_raw_movies.extend(page_results)

print(f"INFO: Successfully retrieved raw movie data for {len(all_raw_movies)} movies.")

INFO: Successfully retrieved raw movie data for 0 movies.


### Transfromation & Relational DMS Pipeline

In [4]:
connection = sqlite3.connect('movies.db')
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS movies (
    title TEXT,
    runtime_minutes INTEGER,
    release_date DATE,
    genres TEXT
);
""")

cursor.execute("DELETE FROM movies;")

for movie in all_raw_movies:
    title = movie.get("title")
    runtime = movie.get("runtime")
    release_date = movie.get("release_date")

    flat_genres = ",".join(movie.get("genres", []))

    cursor.execute("""
        INSERT INTO movies (title, runtime_minutes, release_date, genres)
        VALUES (?, ?, ?, ?)
    """, (title, runtime, release_date, flat_genres))

connection.commit()
connection.close()

print("INFO: Movie data successfully inserted into the database and connection closed.")

INFO: Movie data successfully inserted into the database and connection closed.


### Audit & Integrity Verification

In [5]:
test_connection = sqlite3.connect('movies.db')
test_cursor = test_connection.cursor()

test_cursor.execute("SELECT * FROM movies;")
all_movies = test_cursor.fetchall()

print(f"DATABASE AUDIT SUCCESS: Retrieved {len(all_movies)} records from the 'movies' table.")
print(f"FIRST RECORD SAMPLE: {all_movies[0]}")

test_connection.close()

DATABASE AUDIT SUCCESS: Retrieved 0 records from the 'movies' table.


IndexError: list index out of range